ASX200 Similarity Engine & Snapshot Analysis 

This notebook calculates the rolling Z-scores for 
- Momentum (12-month returns relative to historical mean)
- Volatility (30-day realised volatility relative to historical mean)
- Idiosyncratic Volatility (Residual Alpha after removing market influence) 
- Liquidity Proxy (30-day average daily dollar volume) 

It uses a live market-state 'snapshot' input to query our historical database and find the most similar historical analogues using Euclidean distance. 


In [ ]:
import yfinance as yf 
import pandas as pd
import numpy as np 

In [ ]:
#METRIC CALCULATIONS
def get_current_snapshot(ticker: str) -> dict:
    yt = ticker if ticker.endswith(".AX") else ticker + ".AX"
    market_ticker = "^AXJO"
    
    # Download 2y to have enough history for the Z-score means
    raw = yf.download([yt, market_ticker], period="2y", auto_adjust=True, progress=False)
    
    if raw.empty or yt not in raw['Close']:
        raise ValueError(f"Ticker {yt} data missing.")

    prices = raw["Close"]
    volumes = raw["Volume"]
    
    # Daily Returns
    stock_ret = prices[yt].pct_change().dropna()
    market_ret = prices[market_ticker].pct_change().dropna()

    # 1. Momentum Z-Score
    rolling_12m = stock_ret.rolling(252).apply(lambda x: (1 + x).prod() - 1).dropna()
    momentum_zscore = (rolling_12m.iloc[-1] - rolling_12m.mean()) / rolling_12m.std()

    # 2. Volatility Z-Score
    rolling_vol = stock_ret.rolling(30).std() * np.sqrt(252)
    rolling_vol = rolling_vol.dropna()
    vol_zscore = (rolling_vol.iloc[-1] - rolling_vol.mean()) / rolling_vol.std()

    # 3. Idiosyncratic Volatility (Residual from Market Regression)
    aligned = pd.concat([stock_ret, market_ret], axis=1).dropna()
    aligned.columns = ['stock', 'market']
    X, y = aligned['market'], aligned['stock']
    
    cov_matrix = np.cov(y, X)
    beta = cov_matrix[0, 1] / cov_matrix[1, 1]
    
    alpha = y.mean() - beta * X.mean()
    residuals = y - (alpha + beta * X)
    idio_vol = residuals.std() * np.sqrt(252)

    # 4. Liquidity Proxy (Average Daily Dollar Volume)
    dollar_volume = (prices[yt] * volumes[yt]).rolling(30).mean().iloc[-1]

    return {
        "ticker": ticker,
        "date": str(prices.index[-1].date()),
        "momentum_z": round(float(momentum_zscore), 4),
        "volatility_z": round(float(vol_zscore), 4),
        "idio_vol": round(float(idio_vol), 4),
        "dollar_vol_30d": round(float(dollar_volume), 2)
    }


Portfolio Validation

The following code shows the validation for SIIF's current portfolio (comprised of 10 stocks, as opposed to all ASX200) 

In [ ]:
# SIIF CURRENT PORTFOLIO HOLDINGS
portfolio_tickers = ["PLY", "LAU", "COS", "ANG", "VVA", "WTC", "AUB", "TLX", "XYZ", "DUG"]
results = []

print("--- Extracting Live Metrics for SMIF Portfolio ---")
for t in portfolio_tickers:
    try:
        data = get_current_snapshot(t)
        results.append(data)
        print(f"✅ {t:4} | Snapshot captured.")
    except Exception as e:
        print(f"❌ {t:4} | Error: {e}")

# Results converted to table 
portfolio_df = pd.DataFrame(results)
display(portfolio_df)

portfolio_df.to_csv("portfolio_snapshot.csv", index=False)


FUTURE ROADMAP 

Now that the live metric engine is validated, the next phase (week 2) invol